# Day 8 lab, Parts B and C: a custom detector

**Goal.** You train a YOLO11n model for two classes, a bottle and a cup. You
measure it on test images, and you export it for the Raspberry Pi: to NCNN
and to LiteRT, in `float32` and in `int8`, for two image sizes.

**Deliverable.** The model files in the folder `models/` for the Raspberry
Pi, and the tables of Parts B and C in the file `report.md`.

**Time.** Part B has 50 minutes (sections 0 to 7). The export of Part C
(sections 8 to 10) has 15 minutes. The complete notebook runs in about 13
minutes on 4 cores of the work computer of the course. The training needs
about 9 of these minutes.

**How to work.** Run the cells in order. The training runs in the
background: you start it in section 2, and you do the three tasks while it
runs. A cell with the mark `TODO (student)` has a task. The cell after a
task checks your result. A section that needs a task runs only when the
task is complete.

## 0. Setup

Run this cell first. It loads the packages and sets the paths.

The cell defines these helpers:

- `quiet()`: hides the many status lines that an exporter prints.
- `read_labels(image_path)`: the labels of an image as rows
  `class, x1, y1, x2, y2` in pixels.
- `detections(model, image_path, imgsz, conf, class_map)`: the detections of
  a model as rows `class, score, x1, y1, x2, y2` in pixels.
- `show_boxes(axis, image_path, rows, colour, style)`: draws boxes on an image.
- `validate(path, imgsz)`: the mAP of a model file on the test images.

In [ ]:
import collections
import contextlib
import csv
import glob
import json
import logging
import os
import shutil
import subprocess
import sys
import time
import warnings

warnings.filterwarnings("ignore")
os.environ["YOLO_VERBOSE"] = "False"             # no status lines of the package
os.environ["LITERT_TORCH_SHOW_PROGRESS"] = "n"   # no progress lines of the LiteRT export
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")

import matplotlib.pyplot as plt
import numpy as np
import torch
import ultralytics
from PIL import Image
from ultralytics import YOLO
from ultralytics.data.augment import LetterBox
from ultralytics.utils import ASSETS

# The lab folder. The notebook of the folder solutions/ uses the folder above.
LAB = os.path.abspath("." if os.path.isdir("dataset") else "..")
DATA = os.path.join(LAB, "data", "cup_bottle")
DATA_YAML = os.path.join(DATA, "data.yaml")
MODELS = os.path.join(LAB, "models")
RUNS = os.path.join(LAB, "runs")
os.makedirs(MODELS, exist_ok=True)
os.makedirs(RUNS, exist_ok=True)

NAMES = ["bottle", "cup"]      # class 0 and class 1 of the dataset
COCO_TO_DATASET = {39: 0, 41: 1}   # the numbers of the two classes in the 80 classes of COCO
IMGSZ = 320                    # image size of the training
EPOCHS = 25


@contextlib.contextmanager
def quiet():
    '''Hide the status lines that an exporter prints.'''
    sys.stdout.flush()
    sys.stderr.flush()
    streams = [sys.stdout, sys.stderr]
    saved_write = [stream.write for stream in streams]
    saved_fd = os.dup(1), os.dup(2)
    logging.disable(logging.CRITICAL)
    with open(os.devnull, "w") as null:
        for stream in streams:
            stream.write = lambda *args, **kwargs: 0
        os.dup2(null.fileno(), 1)
        os.dup2(null.fileno(), 2)
        try:
            yield
        finally:
            os.dup2(saved_fd[0], 1)
            os.dup2(saved_fd[1], 2)
            os.close(saved_fd[0])
            os.close(saved_fd[1])
            for stream, write in zip(streams, saved_write):
                stream.write = write
            logging.disable(logging.NOTSET)


def images_of(split):
    '''The image files of a split: "train", "valid", or "test".'''
    return sorted(glob.glob(os.path.join(DATA, split, "images", "*.jpg")))


def read_labels(image_path):
    '''The labels of an image as rows class, x1, y1, x2, y2 in pixels.'''
    width, height = Image.open(image_path).size
    folder, name = os.path.split(image_path)
    path = os.path.join(os.path.dirname(folder), "labels", name[:-4] + ".txt")
    rows = []
    with open(path) as handle:
        for line in handle:
            number, cx, cy, w, h = line.split()
            cx, cy, w, h = float(cx) * width, float(cy) * height, float(w) * width, float(h) * height
            rows.append([int(number), cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2])
    return np.array(rows, dtype=np.float32).reshape(-1, 5)


def detections(model, image_path, imgsz, conf=0.25, class_map=None):
    '''The detections of a model as rows class, score, x1, y1, x2, y2 in pixels.

    class_map: a dictionary from a class of the model to a class of the
    dataset. A detection of a different class is not in the result.
    '''
    result = model.predict(image_path, imgsz=imgsz, conf=conf, verbose=False)[0]
    rows = []
    for number, score, box in zip(result.boxes.cls.tolist(), result.boxes.conf.tolist(),
                                  result.boxes.xyxy.tolist()):
        number = int(number)
        if class_map is not None:
            if number not in class_map:
                continue
            number = class_map[number]
        rows.append([number, score] + box)
    return np.array(rows, dtype=np.float32).reshape(-1, 6)


def show_boxes(axis, image_path, rows, colour, style="-", text=True):
    '''Draw boxes on an image. A row is class, x1, y1, x2, y2, or class, score, x1, y1, x2, y2.'''
    if not axis.images:
        axis.imshow(Image.open(image_path))
        axis.axis("off")
    for row in rows:
        x1, y1, x2, y2 = row[-4:]
        axis.add_patch(plt.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, edgecolor=colour,
                                     linewidth=2, linestyle=style))
        if text:
            label = NAMES[int(row[0])] + (" %.2f" % row[1] if len(row) == 6 else "")
            axis.text(x1, y1, label, color="white", fontsize=7, va="bottom",
                      bbox=dict(facecolor=colour, edgecolor="none", pad=1))


def validate(path, imgsz):
    '''The mAP of a model file on the test images. Each image goes into a square of imgsz pixels.'''
    with quiet():
        metrics = YOLO(path, task="detect").val(
            data=DATA_YAML, split="test", imgsz=imgsz, rect=False, workers=0, plots=False,
            verbose=False, project=RUNS, name="val", exist_ok=True)
    return metrics.box


print("Python %s, torch %s, ultralytics %s" % (
    sys.version.split()[0], torch.__version__.split("+")[0], ultralytics.__version__))
print("Lab folder: %s" % ("solutions/.." if not os.path.isdir("dataset") else "."))

## 1. The dataset (8 minutes)

The kit lab uses its own photos of two objects: a box and a wheel. A
classroom does not have these two objects, so this lab uses two objects
that each classroom has: a **bottle** and a **cup**.

The script `get_dataset.py` downloads 400 images of the dataset COCO and
writes them in the format of the lecture: one folder for each split, and
one text file with the labels for each image. The download has about 60 MB.
A second run downloads nothing.

| Split | Use |
|---|---|
| `train` | The training changes the weights with these images. |
| `valid` | The training measures the model after each epoch with these images, and it keeps the best epoch. The export to `int8` takes its calibration images from this split. |
| `test` | You report the accuracy with these images. No step before the report uses them. |

In [ ]:
reply = subprocess.run([sys.executable, os.path.join(LAB, "get_dataset.py")],
                       capture_output=True, text=True)
lines = reply.stdout.strip().split("\n")
start = next((k for k, line in enumerate(lines) if line.startswith("split")), 0)
print("\n".join(lines[start:]))
if reply.returncode != 0:
    print(reply.stderr[-500:])
    raise RuntimeError("The dataset is not complete. Run this cell again.")

Look at the data before you train. The next cell prints the file
`data.yaml` and the label file of one image. It then counts the labels, and
it gives the median size of a label as a part of the image side.

A row of a label file is `class cx cy w h`: the class number, the centre of
the box, and its size. The four numbers are from 0 to 1, as a part of the
image width and height (Part 2 of the lecture).

In [ ]:
print(open(DATA_YAML).read())
example = images_of("train")[0]
print("Image:", os.path.relpath(example, LAB), Image.open(example).size)
print("Label file:")
print(open(example.replace("images", "labels")[:-4] + ".txt").read())

print("%-6s %7s %7s %7s %16s %18s" % ("split", "images", "bottle", "cup", "labels per image", "median label side"))
for split in ("train", "valid", "test"):
    counts = collections.Counter()
    sides = []
    files = images_of(split)
    for path in files:
        width, height = Image.open(path).size
        for number, x1, y1, x2, y2 in read_labels(path):
            counts[NAMES[int(number)]] += 1
            sides.append(max((x2 - x1) / width, (y2 - y1) / height))
    total = sum(counts.values())
    print("%-6s %7d %7d %7d %16.2f %17.0f%%" % (split, len(files), counts["bottle"], counts["cup"],
                                              total / len(files), 100 * np.median(sides)))

The next cell draws the labels on 8 training images. A bottle is red, and a
cup is blue.

Look at each image, and answer in the report:

1. Does each box touch the object on its four sides?
2. Find one object that looks like a bottle or a cup and has no label. The
   rules of the dataset decide this: in COCO, a wine glass, a bowl, and a
   vase are three other classes.

In [ ]:
figure, axes = plt.subplots(2, 4, figsize=(14, 7))
files = images_of("train")
for axis, path in zip(axes.flat, files[::len(files) // 8]):
    labels = read_labels(path)
    show_boxes(axis, path, labels[labels[:, 0] == 0], "red")
    show_boxes(axis, path, labels[labels[:, 0] == 1], "blue")
    axis.set_title(os.path.basename(path), fontsize=8)
plt.tight_layout()
plt.show()

## 2. Start the training (2 minutes)

The script `train_detector.py` trains the model YOLO11n for the two classes.
It starts from the weights of the 80 classes of COCO (transfer learning).
The model gets a new head for 2 classes, and the training runs for 25 epochs
with images of 320 pixels.

The script uses the option `freeze=10`: the first 10 modules of the model,
which are the backbone, do not change. Only the neck and the head learn. An
epoch is then faster, and the 240 training images cannot damage the features
that the backbone learned from 118 000 images.

The cell first downloads the pre-trained weights `yolo11n.pt` (5.4 MB) into
the folder `models/`. The training needs about 9 minutes on 4 cores of a
laptop. The cell starts it **in the background** and returns at once. Continue with section 3 while
it runs. The file `runs/train_log.txt` has the output of the script.

A second run of this cell does not train again when the folder has a trained
model. Set `RETRAIN = True` to train again, for example with more epochs.

On Colab with a hardware accelerator, add the two arguments
`"--device", "0"` to the command. The training is then much faster.

In [ ]:
RETRAIN = False

# The pre-trained model with the 80 classes of COCO. Sections 4 and 5 use it.
with quiet():
    pretrained = YOLO(os.path.join(MODELS, "yolo11n.pt"))

weights = os.path.join(MODELS, "cupbottle.pt")
done_file = os.path.join(RUNS, "cupbottle", "done.json")
training = None
trained = json.load(open(done_file)) if os.path.exists(done_file) else None
if not os.path.exists(weights):
    start = True
elif trained is None:
    start = False       # the file models/cupbottle.pt comes from a different place
else:
    start = (trained["epochs"], trained["imgsz"]) != (EPOCHS, IMGSZ)
if RETRAIN or start:
    log = open(os.path.join(RUNS, "train_log.txt"), "w")
    training = subprocess.Popen(
        [sys.executable, "train_detector.py", "--epochs", str(EPOCHS), "--imgsz", str(IMGSZ)],
        cwd=LAB, stdout=log, stderr=subprocess.STDOUT,
        env=dict(os.environ, YOLO_VERBOSE="True"))
    print("The training runs in the background: %d epochs, image size %d." % (EPOCHS, IMGSZ))
    print("Continue with section 3. Section 6 waits for the end of the training.")
elif trained is None:
    print("The folder has the file models/cupbottle.pt, and no training run of this folder.")
    print("The notebook uses this file. Set RETRAIN = True to train your own model.")
else:
    print("The folder has a trained model: %d epochs, image size %d, %.1f minutes."
          % (trained["epochs"], trained["imgsz"], trained["minutes"]))
    print("The training does not start again. Set RETRAIN = True to train again.")

## 3. Task 1: the IoU of two boxes (7 minutes)

The IoU (intersection over union) measures how well two boxes agree: the
area that the two boxes share, divided by the area that the two boxes cover
together. It is 1 for two equal boxes and 0 for two boxes that do not touch.
The suppression and the metrics use it (Parts 1 and 2 of the lecture).

**Task 1.** Complete the function `box_iou`. A box is `x1, y1, x2, y2`: the
top left corner and the bottom right corner in pixels.

1. The intersection is a rectangle. Its left side is the larger of the two
   `x1` values, and its right side is the smaller of the two `x2` values. Do
   the same for `y`. If the width or the height is negative, the boxes do
   not touch: use 0.
2. The union is the area of box `a`, plus the area of box `b`, minus the
   intersection.
3. Return the intersection divided by the union.

In [ ]:
def box_iou(a, b):
    '''The IoU of two boxes. A box is x1, y1, x2, y2.'''
    # TODO (student), Task 1: write the three steps of the text above.
    return None

The next cell checks your function with the box pairs of the exercise of
the lecture, and with 200 random box pairs.

In [ ]:
def _iou_matrix(boxes_a, boxes_b):
    '''The IoU of each box of a list with each box of a second list (the code of the check).'''
    boxes_a, boxes_b = np.asarray(boxes_a, dtype=np.float64), np.asarray(boxes_b, dtype=np.float64)
    x1 = np.maximum(boxes_a[:, None, 0], boxes_b[None, :, 0])
    y1 = np.maximum(boxes_a[:, None, 1], boxes_b[None, :, 1])
    x2 = np.minimum(boxes_a[:, None, 2], boxes_b[None, :, 2])
    y2 = np.minimum(boxes_a[:, None, 3], boxes_b[None, :, 3])
    inter = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)
    area_a = (boxes_a[:, 2] - boxes_a[:, 0]) * (boxes_a[:, 3] - boxes_a[:, 1])
    area_b = (boxes_b[:, 2] - boxes_b[:, 0]) * (boxes_b[:, 3] - boxes_b[:, 1])
    return inter / (area_a[:, None] + area_b[None, :] - inter)


pairs = [
    ("Pair 1 of the lecture", (0, 0, 10, 10), (2, 0, 12, 10)),
    ("Pair 2 of the lecture", (0, 0, 4, 4), (2, 0, 6, 4)),
    ("The second box for pair 1", (0, 0, 10, 10), (1, 0, 11, 10)),
    ("Two boxes that do not touch", (0, 0, 4, 4), (6, 6, 9, 9)),
    ("Two equal boxes", (3, 2, 8, 9), (3, 2, 8, 9)),
]
task1_complete = True
for text, a, b in pairs:
    value = box_iou(a, b)
    expected = _iou_matrix([a], [b])[0, 0]
    correct = value is not None and abs(value - expected) < 1e-6
    task1_complete &= correct
    print("%-28s IoU %s   %s" % (text, "none" if value is None else "%.2f" % value,
                                 "correct" if correct else "expected %.2f" % expected))
rng = np.random.default_rng(0)
corners = rng.uniform(0, 100, size=(200, 2, 2))
sizes = rng.uniform(5, 60, size=(200, 2, 2))
for k in range(200):
    a = (*corners[k, 0], *(corners[k, 0] + sizes[k, 0]))
    b = (*corners[k, 1], *(corners[k, 1] + sizes[k, 1]))
    value = box_iou(a, b)
    task1_complete &= value is not None and abs(value - _iou_matrix([a], [b])[0, 0]) < 1e-6
print("Task 1:", "complete" if task1_complete else "not complete")

## 4. Task 2: non-maximum suppression (8 minutes)

The model YOLO11n gives 8400 candidates for an image of 640 pixels. Each
candidate is a box with 80 class scores. Many candidates show the same
object. The next cell runs the pre-trained model of section 2 on the image
`bus.jpg` of the package, and keeps the candidates with a score above 0.25.
This is the image of Part 1 of the lecture.

In [ ]:
bus = str(ASSETS / "bus.jpg")

# Pre-processing: the image goes into a square of 640 pixels with grey borders.
square = LetterBox(new_shape=(640, 640), auto=False)(image=np.array(Image.open(bus))[:, :, ::-1])
tensor = torch.from_numpy(square[:, :, ::-1].transpose(2, 0, 1).copy()).float().div(255)[None]
with torch.no_grad():
    raw = pretrained.model.eval()(tensor)[0][0].numpy()
print("Raw output:", raw.shape, "(4 box values and 80 class scores, for each of 8400 candidates)")

# Decode: the best class of each candidate, and the box as x1, y1, x2, y2.
candidate_scores = raw[4:].max(0)
candidate_classes = raw[4:].argmax(0)
mask = candidate_scores > 0.25
cx, cy, w, h = raw[:4, mask]
candidate_boxes = np.stack([cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2], 1)
candidate_scores, candidate_classes = candidate_scores[mask], candidate_classes[mask]
print("Candidates with a score above 0.25: %d" % len(candidate_scores))
for number, count in sorted(collections.Counter(candidate_classes.tolist()).items()):
    print("  %-8s %d candidates" % (pretrained.names[number], count))

**Task 2.** Complete the function `nms`. It gets the boxes and the scores of
the candidates of **one class**. It returns the indices of the boxes that
stay.

1. Sort the indices by the score, the highest score first.
2. Take the first index of the list, and add it to the result.
3. Remove from the list each box that has an IoU above the threshold with
   this box. Use your function `box_iou`.
4. Do steps 2 and 3 again until the list is empty.

In [ ]:
def nms(boxes, scores, iou_threshold):
    '''Non-maximum suppression for the candidates of one class.

    boxes: array (N, 4) with x1, y1, x2, y2. scores: array (N,).
    Return the list of the indices that stay, the highest score first.
    '''
    # TODO (student), Task 2: write the four steps of the text above.
    return None

The next cell applies your function to each class of the image, for two IoU
thresholds. It compares the result with the code of the check. The lecture
measured 5 detections for the threshold 0.7.

In [ ]:
def _nms_reference(boxes, scores, iou_threshold):
    '''The code of the check. It does not use a function of a task.'''
    overlap = _iou_matrix(boxes, boxes)
    order = list(np.argsort(-scores))
    keep = []
    while order:
        best = order.pop(0)
        keep.append(int(best))
        order = [i for i in order if overlap[best, i] <= iou_threshold]
    return keep


def suppress(function, iou_threshold):
    '''Apply a suppression function to each class. Return the indices that stay, or None.'''
    keep = []
    for number in np.unique(candidate_classes):
        index = np.flatnonzero(candidate_classes == number)
        kept = function(candidate_boxes[index], candidate_scores[index], iou_threshold)
        if kept is None:
            return None
        keep += [int(index[k]) for k in kept]
    return sorted(keep, key=lambda k: -candidate_scores[k])


task2_complete = True
for threshold in (0.7, 0.9):
    kept = suppress(nms, threshold)
    expected = suppress(_nms_reference, threshold)
    correct = kept is not None and list(kept) == list(expected)
    task2_complete &= correct
    print("IoU threshold %.1f: %s detections, expected %d" % (
        threshold, "no" if kept is None else len(kept), len(expected)))
    if correct:
        for k in kept:
            print("   %-8s %.2f" % (pretrained.names[int(candidate_classes[k])], candidate_scores[k]))
print("Task 2:", "complete" if task2_complete else "not complete")

The figure shows the candidates before the suppression, and the detections
after it. The image is the square of 640 pixels that the model gets.

In [ ]:
kept = suppress(nms if task2_complete else _nms_reference, 0.7)
figure, axes = plt.subplots(1, 2, figsize=(11, 5.5))
for axis, index, title in ((axes[0], range(len(candidate_scores)), "%d candidates" % len(candidate_scores)),
                           (axes[1], kept, "%d detections, IoU threshold 0.7" % len(kept))):
    axis.imshow(square[:, :, ::-1])
    axis.axis("off")
    axis.set_title(title)
    for k in index:
        x1, y1, x2, y2 = candidate_boxes[k]
        colour = "orange" if pretrained.names[int(candidate_classes[k])] == "bus" else "cyan"
        axis.add_patch(plt.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, edgecolor=colour, linewidth=1))
plt.tight_layout()
plt.show()

## 5. Task 3: true positives, false positives, false negatives (10 minutes)

A detection is a **true positive** (TP) when it has an IoU of 0.5 or more
with a label of the same class, and no detection with a higher score took
that label. A detection with no label is a **false positive** (FP). A label
with no detection is a **false negative** (FN). Part 2 of the lecture gives
the method.

**Task 3.** Complete the function `count_image`. It gets the boxes of the
detections of **one class** in **one image**, sorted by the score with the
highest score first, and the boxes of the labels of that class.

1. Make the list of the free labels. At the start, each label is free.
2. For each detection, in the order of the list: find the free label with
   the highest IoU. Use your function `box_iou`.
3. If this IoU is the threshold or more, the detection is a true positive,
   and the label is not free now.
4. Return three numbers: the true positives, the false positives (the
   detections that are not true positives), and the false negatives (the
   labels that are free at the end).

In [ ]:
def count_image(detection_boxes, label_boxes, iou_threshold=0.5):
    '''Count TP, FP, and FN for one class in one image.

    detection_boxes: array (N, 4), the highest score first. label_boxes: array (M, 4).
    Return the tuple (TP, FP, FN).
    '''
    # TODO (student), Task 3: write the four steps of the text above.
    return None

The next cell checks your function with five small cases. The first three
cases are the boxes of the exercise of the lecture.

In [ ]:
A = np.array
cases = [
    ("Pair 1: one detection, IoU 0.67", A([[2, 0, 12, 10]]), A([[0, 0, 10, 10]]), (1, 0, 0)),
    ("Pair 2: one detection, IoU 0.33", A([[2, 0, 6, 4]]), A([[0, 0, 4, 4]]), (0, 1, 1)),
    ("Pair 1 with a second box of a lower score", A([[2, 0, 12, 10], [1, 0, 11, 10]]),
     A([[0, 0, 10, 10]]), (1, 1, 0)),
    ("Two labels, one detection", A([[20, 20, 30, 30]]), A([[0, 0, 10, 10], [20, 20, 30, 31]]), (1, 0, 1)),
    ("No detection, two labels", np.zeros((0, 4)), A([[0, 0, 10, 10], [20, 20, 30, 30]]), (0, 0, 2)),
]
task3_complete = True
for text, boxes, labels, expected in cases:
    value = count_image(boxes.astype(float), labels.astype(float))
    correct = value is not None and tuple(int(v) for v in value) == expected
    task3_complete &= correct
    print("%-44s %-12s %s" % (text, "none" if value is None else tuple(int(v) for v in value),
                              "correct" if correct else "expected %s" % (expected,)))
print("Task 3:", "complete" if task3_complete else "not complete")

### The baseline: the pre-trained model

The pre-trained model knows 80 classes. Two of them are `bottle` (class 39)
and `cup` (class 41). Before you look at your own model, measure this
baseline: how well does the model of the package find the two classes in the
100 test images?

The next cell runs the pre-trained model on each test image, for two image
sizes. It keeps the detections of the two classes with a score of 0.25 or
more, and it counts with your function. The cell needs about 30 seconds,
and more when the training still runs.

- Precision = TP / (TP + FP): the part of the detections that is correct.
- Recall = TP / (TP + FN): the part of the labels that the model finds.

Write your prediction in the report before you run the cell: is the recall
at 320 pixels lower than at 640 pixels?

In [ ]:
def evaluate(model, imgsz, class_map=None, conf=0.25):
    '''Count TP, FP, and FN for each class on the test images, with the function of Task 3.'''
    counts = np.zeros((len(NAMES), 3), dtype=int)
    for path in images_of("test"):
        labels = read_labels(path)
        rows = detections(model, path, imgsz, conf, class_map)
        rows = rows[np.argsort(-rows[:, 1])]            # the highest score first
        for number in range(len(NAMES)):
            counts[number] += count_image(rows[rows[:, 0] == number, 2:],
                                          labels[labels[:, 0] == number, 1:])
    return counts


def print_counts(title, counts):
    print(title)
    print("  %-8s %5s %5s %5s %10s %8s" % ("class", "TP", "FP", "FN", "precision", "recall"))
    rows = [(NAMES[k], counts[k]) for k in range(len(NAMES))] + [("all", counts.sum(0))]
    for name, (tp, fp, fn) in rows:
        print("  %-8s %5d %5d %5d %10.2f %8.2f" % (name, tp, fp, fn, tp / max(tp + fp, 1), tp / max(tp + fn, 1)))


results = {}
if task1_complete and task3_complete:
    for size in (320, 640):
        results["pre-trained", size] = evaluate(pretrained, size, COCO_TO_DATASET)
        print_counts("Pre-trained YOLO11n (80 classes), image size %d, score threshold 0.25:" % size,
                     results["pre-trained", size])
else:
    print("This section needs Task 1 and Task 3.")

## 6. The result of the training (8 minutes)

The next cell waits for the end of the training. It prints one line for
each epoch: the time, the two losses of the training images, and the mAP on
the 60 validation images.

- `box loss`: the error of the place and the size of the boxes.
- `class loss`: the error of the class scores.
- `mAP50`: the mean AP of the two classes at the IoU threshold 0.5.
- `mAP50-95`: the mean for the IoU thresholds 0.50 to 0.95.

In [ ]:
def read_results():
    '''The rows of the file results.csv, which the package writes after each epoch.'''
    path = os.path.join(RUNS, "cupbottle", "results.csv")
    if not os.path.exists(path):
        return []
    with open(path) as handle:
        return [{key.strip(): float(value) for key, value in row.items()} for row in csv.DictReader(handle)]


def print_epochs(rows, start=0):
    for row in rows[start:]:
        print("%5d %7.0f %9.3f %11.3f %7.3f %9.3f" % (
            row["epoch"], row["time"], row["train/box_loss"], row["train/cls_loss"],
            row["metrics/mAP50(B)"], row["metrics/mAP50-95(B)"]))


print("%5s %7s %9s %11s %7s %9s" % ("epoch", "time s", "box loss", "class loss", "mAP50", "mAP50-95"))
if training is not None:
    shown = 0
    while training.poll() is None:
        try:
            rows = read_results()       # the package can write the file at this moment
        except ValueError:
            rows = []
        print_epochs(rows, shown)
        shown = max(shown, len(rows))
        time.sleep(5)
    print_epochs(read_results(), shown)
    if training.returncode != 0:
        print("ERROR: the training stopped with an error. Read the file runs/train_log.txt.")
else:
    print_epochs(read_results())

history = read_results()
trained = json.load(open(done_file)) if os.path.exists(done_file) else None
if not history:
    print("This folder has no training run. The notebook uses the given file models/cupbottle.pt.")
if trained and history:
    best = max(history, key=lambda row: row["metrics/mAP50-95(B)"])
    print("\nTraining time: %.1f minutes for %d epochs." % (trained["minutes"], trained["epochs"]))
    print("Best epoch: %d, with mAP50 %.3f and mAP50-95 %.3f on the validation images."
          % (best["epoch"], best["metrics/mAP50(B)"], best["metrics/mAP50-95(B)"]))
    print("Weights of the best epoch: models/cupbottle.pt, %d bytes." % os.path.getsize(weights))

The curves show the same values. Answer in the report:

1. Do the two losses still go down at the last epoch?
2. Does the mAP still go up at the last epoch? Would more epochs help?
3. The curve of the mAP is not smooth. The validation set has only 60
   images. How does this explain the curve?

The package also writes figures into the folder `runs/cupbottle/`: the
precision-recall curve, the confusion matrix, and some validation images
with the predictions. Open the files `results.png` and `val_batch0_pred.jpg`.

In [ ]:
if history:
    epochs = [row["epoch"] for row in history]
    figure, axes = plt.subplots(1, 2, figsize=(11, 3.6))
    axes[0].plot(epochs, [row["train/box_loss"] for row in history], label="box loss")
    axes[0].plot(epochs, [row["train/cls_loss"] for row in history], label="class loss")
    axes[0].set_title("Loss on the training images")
    axes[1].plot(epochs, [row["metrics/mAP50(B)"] for row in history], label="mAP50")
    axes[1].plot(epochs, [row["metrics/mAP50-95(B)"] for row in history], label="mAP50-95")
    axes[1].set_title("mAP on the validation images")
    axes[1].set_ylim(0, 1)
    for axis in axes:
        axis.set_xlabel("epoch")
        axis.grid(alpha=0.3)
        axis.legend()
    plt.tight_layout()
    plt.show()

## 7. Your model on the test images (7 minutes)

The validation images selected the best epoch, so they are not a fair test.
The next cell measures the model on the 100 test images, which no step used
before. The function `validate` of the setup cell calls the validation of
the package, which gives the output of Part 2 of the lecture.

In [ ]:
if os.path.exists(weights):
    custom = YOLO(weights)
    box = validate(weights, IMGSZ)
    print("Your model on the 100 test images, image size %d:" % IMGSZ)
    print("  %-8s %10s %8s %7s %9s" % ("class", "precision", "recall", "mAP50", "mAP50-95"))
    print("  %-8s %10.3f %8.3f %7.3f %9.3f" % ("all", box.mp, box.mr, box.map50, box.map))
    for k, number in enumerate(box.ap_class_index):
        print("  %-8s %10.3f %8.3f %7.3f %9.3f" % (NAMES[int(number)], box.p[k], box.r[k], box.ap50[k], box.ap[k]))
    test_map = {"all": (float(box.map50), float(box.map))}
else:
    custom = None
    print("The file models/cupbottle.pt does not exist. Read the file runs/train_log.txt.")

The package selects a score threshold for its precision and its recall. The
next cell counts with your function at the score threshold 0.25, the same
method as for the baseline. It then compares your model with the baseline.

Write your prediction in the report before you run the cell: your model
has 2 classes and saw 240 images. The pre-trained model has 80 classes and
saw 118 000 images, with many bottles and cups. Which model has the higher
recall for the two classes?

In [ ]:
if custom is not None and task1_complete and task3_complete:
    results["your model", IMGSZ] = evaluate(custom, IMGSZ)
    print_counts("Your model (2 classes), image size %d, score threshold 0.25:" % IMGSZ,
                 results["your model", IMGSZ])
    print()
    print("%-30s %5s %5s %5s %10s %8s" % ("model, both classes", "TP", "FP", "FN", "precision", "recall"))
    for (name, size), counts in results.items():
        tp, fp, fn = counts.sum(0)
        print("%-30s %5d %5d %5d %10.2f %8.2f" % ("%s at %d" % (name, size), tp, fp, fn,
                                                  tp / max(tp + fp, 1), tp / max(tp + fn, 1)))
else:
    print("This section needs the trained model, Task 1, and Task 3.")

The figure shows 8 test images. A label is a dashed green box. A detection
of your model is a red box (bottle) or a blue box (cup), with its score.
Find one false positive and one false negative, and describe them in the
report.

In [ ]:
if custom is not None:
    figure, axes = plt.subplots(2, 4, figsize=(14, 7))
    files = images_of("test")
    for axis, path in zip(axes.flat, files[::len(files) // 8]):
        rows = detections(custom, path, IMGSZ)
        show_boxes(axis, path, read_labels(path), "lime", style="--", text=False)
        show_boxes(axis, path, rows[rows[:, 0] == 0], "red")
        show_boxes(axis, path, rows[rows[:, 0] == 1], "blue")
        axis.set_title(os.path.basename(path), fontsize=8)
    plt.tight_layout()
    plt.show()

**End of Part B.** The comparison is a normal result: 240 images do not beat
a model that learned the two classes from a much larger dataset. A custom
model is the correct tool when the pre-trained model does not know your
classes (the box and the wheel of the kit lab), or when your images are
different from the images of COCO (a different camera position, a different
light). With more images of your own camera, the custom model becomes better
than the baseline.

## 8. Part C: export (8 minutes)

The Raspberry Pi does not run the PyTorch file in the lab. The next cell
exports your model for two runtimes and two image sizes:

| File | Runtime | Precision |
|---|---|---|
| `cupbottle_SIZE_ncnn_model/` (a folder) | NCNN | `float32` |
| `cupbottle_SIZE.tflite` | LiteRT | `float32` |
| `cupbottle_SIZE_int8.tflite` | LiteRT | `int8` weights and `int8` activations |

`SIZE` is 320 or 640. An exported file has a static input shape: the image
size is a part of the file (Part 3 of the lecture).

The export to `int8` needs calibration images. The argument `data` names the
dataset, and the export takes the 60 images of the split `valid`.

The cell needs about 2 minutes. Write your prediction in the report before
you run the cell: how large is the `int8` file, as a part of the `float32`
file?

The command line of the package does the same work:

```bash
yolo export model=models/cupbottle_320.pt format=ncnn imgsz=320
yolo export model=models/cupbottle_320.pt format=litert imgsz=320
yolo export model=models/cupbottle_320.pt format=litert imgsz=320 quantize=8 data=data/cup_bottle/data.yaml
```

In [ ]:
EXPORT_SIZES = [320, 640]


def folder_bytes(path):
    '''The size of a file, or of the two model files of an NCNN folder.'''
    if os.path.isfile(path):
        return os.path.getsize(path)
    return sum(os.path.getsize(name) for name in glob.glob(os.path.join(path, "model.ncnn.*")))


exported = []       # (file, runtime, precision, image size)
if os.path.exists(weights):
    print("%-28s %-8s %-9s %5s %12s %8s" % ("file", "runtime", "precision", "size", "bytes", "time s"))
    for size in EXPORT_SIZES:
        # The export writes its files next to the PyTorch file, with its name.
        source = os.path.join(MODELS, "cupbottle_%d.pt" % size)
        shutil.copy(weights, source)
        jobs = [
            ("NCNN", "float32", dict(format="ncnn")),
            ("LiteRT", "float32", dict(format="litert")),
            ("LiteRT", "int8", dict(format="litert", quantize=8, data=DATA_YAML)),
        ]
        for runtime, precision, arguments in jobs:
            start = time.time()
            with quiet():
                path = YOLO(source).export(imgsz=size, **arguments)
            path = os.path.join(MODELS, os.path.basename(os.path.normpath(str(path))))
            exported.append((path, runtime, precision, size))
            print("%-28s %-8s %-9s %5d %12d %8.0f" % (
                os.path.basename(path), runtime, precision, size, folder_bytes(path), time.time() - start))
    print("%-28s %-8s %-9s %5s %12d" % ("cupbottle.pt", "PyTorch", "float16", "all", os.path.getsize(weights)))
else:
    print("The file models/cupbottle.pt does not exist.")

## 9. The accuracy of each exported file (5 minutes)

A frame rate has no value without the accuracy of the same file (Part 3 of
the lecture). The next cell measures each LiteRT file on the 100 test
images. It also measures the PyTorch file at the two image sizes.

The NCNN file has the same `float32` weights as the LiteRT `float32` file.
The lecture measured the same mAP for the two files, so the cell does not
measure the NCNN file.

The cell needs about 30 seconds. Write your predictions in the report
before you run it:

1. How many points of mAP50-95 does the `int8` file lose?
2. Your model learned from images of 320 pixels. Is its mAP at 640 pixels
   higher, the same, or lower?

In [ ]:
accuracy = {}
if exported:
    print("%-28s %-9s %5s %7s %9s" % ("file", "precision", "size", "mAP50", "mAP50-95"))
    for size in EXPORT_SIZES:
        box = validate(os.path.join(MODELS, "cupbottle_%d.pt" % size), size)
        accuracy["PyTorch", size] = (float(box.map50), float(box.map))
        print("%-28s %-9s %5d %7.3f %9.3f" % ("cupbottle_%d.pt" % size, "PyTorch", size, box.map50, box.map))
        for path, runtime, precision, file_size in exported:
            if runtime == "LiteRT" and file_size == size:
                box = validate(path, size)
                accuracy[precision, size] = (float(box.map50), float(box.map))
                print("%-28s %-9s %5d %7.3f %9.3f" % (os.path.basename(path), precision, size, box.map50, box.map))
    print()
    for size in EXPORT_SIZES:
        loss50 = accuracy["float32", size][0] - accuracy["int8", size][0]
        loss = accuracy["float32", size][1] - accuracy["int8", size][1]
        print("Loss of int8 against float32 at %d pixels: %.3f of mAP50, %.3f of mAP50-95." % (size, loss50, loss))

Write the table and the two losses in the report. The mAP of a small test
set is not exact: with 100 images, a difference of 0.02 is not a clear
result. The lecture measured a loss of 0.093 of mAP50-95 for the complete
model on 500 images of COCO. A loss of this size is a normal result for a
detector with `int8` activations: one scale holds the boxes and the scores.

## 10. The files for the Raspberry Pi

The folder `models/` now has the files for Parts C and D on the board.
Copy them from the laptop, in the folder `Labs/day08/`. `NN` is the number
of your group:

```bash
scp -r models/cupbottle_*_ncnn_model models/cupbottle_*.tflite edge@pi-NN.local:~/edgeai/day08/models/
```

Continue with Part C in the file `README.md`.

In [ ]:
print("%-28s %12s" % ("file for the Raspberry Pi", "bytes"))
for path, runtime, precision, size in exported:
    print("%-28s %12d" % (os.path.basename(path), folder_bytes(path)))

complete = [str(k) for k, flag in ((1, task1_complete), (2, task2_complete), (3, task3_complete)) if flag]
print("\nTasks complete:", ", ".join(complete) if complete else "none")